# 03 — Bot / Fake Account Detector

Train and evaluate the LightGBM + MLP soft-vote ensemble for bot detection. Includes feature importance analysis and ROC/PR curve evaluation.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import torch
import torch.nn as nn
from sklearn.metrics import roc_curve, auc, precision_recall_curve, classification_report, confusion_matrix
import seaborn as sns
from services.ml.app.models.bot_detector import BotMLP
from services.ml.training.generate_synthetic_data import generate_bot_data

device = 'cuda' if torch.cuda.is_available() else 'cpu'
FEATURE_NAMES = ['log_followers', 'log_following', 'log_posts', 'ff_ratio', 'er_deviation', 'engagement_rate']

In [ ]:
X, y = generate_bot_data(n=20000)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Positive (bot) rate: {y_test.mean():.2%}')

In [ ]:
train_ds = lgb.Dataset(X_train, label=y_train)
val_ds   = lgb.Dataset(X_test,  label=y_test, reference=train_ds)

params = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 63, 'learning_rate': 0.05,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'verbose': -1, 'n_jobs': -1,
}
gbm = lgb.train(params, train_ds, num_boost_round=300,
                valid_sets=[val_ds],
                callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(50)])
lgb_preds = gbm.predict(X_test)
print(f'LightGBM AUC: {auc(*roc_curve(y_test, lgb_preds)[:2]):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
lgb.plot_importance(gbm, ax=ax, feature_name=FEATURE_NAMES, importance_type='gain')
ax.set_title('LightGBM Feature Importance (Gain)')
plt.tight_layout()
plt.show()

In [ ]:
X_t = torch.tensor(X_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.float32)
from torch.utils.data import TensorDataset, DataLoader
ds  = TensorDataset(X_t, y_t)
ldr = DataLoader(ds, batch_size=256, shuffle=True)

mlp     = BotMLP().to(device).train()
opt     = torch.optim.Adam(mlp.parameters(), lr=5e-4, weight_decay=1e-4)
loss_fn = nn.BCELoss()

for epoch in range(40):
    for xb, yb in ldr:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss_fn(mlp(xb), yb).backward()
        opt.step()
    if (epoch + 1) % 10 == 0:
        print(f'MLP epoch {epoch+1}/40')

mlp.eval()
with torch.no_grad():
    mlp_preds = mlp(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

ensemble_preds = (lgb_preds + mlp_preds) / 2.0
print(f'Ensemble AUC: {auc(*roc_curve(y_test, ensemble_preds)[:2]):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for preds, name, color in [
    (lgb_preds, 'LightGBM', '#405DE6'),
    (mlp_preds, 'MLP', '#833AB4'),
    (ensemble_preds, 'Ensemble', '#E1306C'),
]:
    fpr, tpr, _ = roc_curve(y_test, preds)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.3f})', linewidth=2)
    prec, rec, _ = precision_recall_curve(y_test, preds)
    axes[1].plot(rec, prec, label=name, linewidth=2)

axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
plt.tight_layout()
plt.show()

print('\nEnsemble Classification Report:')
print(classification_report(y_test, (ensemble_preds >= 0.5).astype(int), target_names=['normal', 'bot']))